In [0]:
# Python library used to send HTTP requests
import requests

# Python library used to parse and format JSON text strings
import json

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Python library for date functions
from datetime import datetime, timezone, timedelta
from dateutil.relativedelta import relativedelta

# =====================================================
# 1. Retry behavior
# =====================================================

error_codes = {
    "service_unavailable": 503,
    "gateway_timeout": 504,
    "too_many_requests": 429,
    "internal_service_error": 500,
    "bad_gateway": 502
}

retry_strategy = Retry(
    total=5,
    connect=5,
    read=5,
    status=5,
    backoff_factor=1,
    status_forcelist=list(error_codes.values()),
    allowed_methods=["GET"],
    respect_retry_after_header=True,
    raise_on_status=False
)

session = requests.Session()

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

# =====================================================
# 2. Request open-meteo API
# =====================================================

url = "https://archive-api.open-meteo.com/v1/archive"

city_configs = {
    "Frisco": {"latitude": 33.1507, "longitude": -96.8236},
    "Plano": {"latitude": 33.0198, "longitude": -96.6989},
    "McKinney": {"latitude": 33.1972, "longitude": -96.6398},
    "Prosper": {"latitude": 33.2362, "longitude": -96.8017},
    "Little Elm": {"latitude": 33.1646, "longitude": -96.9372}
}

city_names = list(city_configs.keys())

# Dynamically calculate the start & end date

# ERA5 is preferred for long-term climate data
historical_model = "ERA5"

# ERA5 has a 5 day delay
model_archiving_delay_days = 5

# Shift N days back to account for Open-Meteo's background archiving delay lag
latest_available_date = datetime.now(timezone.utc).date() - timedelta(days=model_archiving_delay_days)

# Shift a small 5 years back to reflect immediate local climate
# Use relative delta for 5 calender years, because leap years may not be accounted for
start_historical_date = latest_available_date - relativedelta(years=5)

# Format the dates into standard API text strings (YYYY-MM-DD)
end_date = latest_available_date.isoformat()
start_date = start_historical_date.isoformat()

print(f"Production pipeline initiated. Rolling window: {start_date} to {end_date}")

params = {
    "latitude": [city_configs[city]["latitude"] for city in city_configs],
    "longitude": [city_configs[city]["longitude"] for city in city_configs],
    "start_date": start_date,
    "end_date": end_date,
    "timezone": "UTC", # Open-meteo may label responses with GMT, but they have a zero UTC offset
    "model": historical_model,
    "hourly": ["temperature_2m", "precipitation"]
}

# Sends a 'GET' request to the specified URL with parameters and stores the response (weather data)
response = session.get(url, params=params, timeout=30)

# Throws exception if HTTP response failed
response.raise_for_status()

# .json() will extract just the raw text from the response and parse it into a Python dictionary
raw_data = response.json()

# =====================================================
# 3. Validate top-level response
# =====================================================

expected_city_count = len(city_configs)

# Prevent taking incorrect object types as a valid response
if not isinstance(raw_data, list):
    raise ValueError(f"Expected a list, but recieved a {type(raw_data)}")

# Prevent taking fewer or more cities than requested
if len(raw_data) != expected_city_count:
    raise ValueError(f"Expected {expected_city_count} cities, but recieved {len(raw_data)}")

# =====================================================
# 4. Validate each location
# =====================================================

required_hourly_fields = ["time", "temperature_2m", "precipitation"]

for i, city_data in enumerate(raw_data):
    # Verify city_data is a dictionary
    if not isinstance(city_data, dict):
        raise ValueError(f"Expected a dictionary, but recieved a {type(city_data)}")

    city_name = city_names[i]

    # Verify hourly data exists in city data
    if "hourly" not in city_data:
        raise ValueError(f"Missing 'hourly' data in {city_name}")
    
    hourly = city_data["hourly"]
    missing_fields = set(required_hourly_fields) - set(hourly.keys())

    # Verify all required fields are present in hourly data
    if missing_fields:
        raise ValueError(f"{city_name} is missing fields: {missing_fields}")

    lengths = {field: len(hourly[field]) for field in required_hourly_fields}

    # Verify all required fields have the same length
    if len(set(lengths.values())) != 1:
        raise ValueError(f"Inconsistent hourly data lengths for {city_name}: {lengths}")

# =====================================================
# 5. Preview raw data
# =====================================================

for i, city_data in enumerate(raw_data):
    city_name = city_names[i]
    
    print(f"=== Raw JSON Data Snippet for: {city_name} ===")
    
    # Take just the first 3 items of the hourly data
    preview_dict = {
        "city": city_name,
        "time": city_data["hourly"]["time"][:3], # [:3] means get only the first 3 elements
        "temperature_2m": city_data["hourly"]["temperature_2m"][:3],
        "precipitation": city_data["hourly"]["precipitation"][:3]
    }
    
    print(json.dumps(preview_dict, indent=2))
    print("\n" + "="*40 + "\n")

print(f"Successfully processed sliding history matrix for {len(raw_data)} cities.")

In [0]:
from uuid import uuid4
from datetime import datetime, timezone

from pyspark.sql.types import (
    StructType,
    StructField,
    DoubleType,
    StringType,
    TimestampType
)

# =====================================================
# Write delta bronze table
# =====================================================

ingestion_id = str(uuid4())
ingested_at = datetime.now(timezone.utc)

bronze_records = []

# Store metadata for each city data

for i, city_data in enumerate(raw_data):
    city_name = city_names[i]
    config = city_configs[city_name]

    bronze_records.append({
        "ingestion_id": ingestion_id,
        "ingested_at": ingested_at,
        "source": "open-meteo",
        "endpoint": url,
        "model": historical_model,
        "city": city_name,
        "latitude": config["latitude"],
        "longitude": config["longitude"],
        "request_start_date": start_date,
        "request_end_date": end_date,
        "raw_response": json.dumps(city_data), # Store as string for absolute source fidelity
    })

# An explicit schema is created for the metadata
# The raw response is stored as a string for absolute source fidelity
# The bronze table stores one row per city, metadata, and exact API response

bronze_schema = StructType([
    StructField("ingestion_id", StringType(), False),
    StructField("ingested_at", TimestampType(), False),
    StructField("source", StringType(), False),
    StructField("endpoint", StringType(), False),
    StructField("model", StringType(), False),
    StructField("city", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("request_start_date", StringType(), False),
    StructField("request_end_date", StringType(), False),
    StructField("raw_response", StringType(), False)
])

df_bronze = spark.createDataFrame(bronze_records, schema=bronze_schema)

display(df_bronze)

(
    df_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("weather_project.north_texas_weather.bronze_weather_raw")
)

In [0]:
from pyspark.sql.functions import current_timestamp, explode, arrays_zip, col

# Converts the dictionary into a Spark data frame, which is just a table (rows, columns with strict data types, like an MS excel spreadsheet)
# Currently this only has one row of a big list of temperatures/times, because that's how it was structured originally in the JSON data.
dataframe = spark.createDataFrame(combined_raw_data)

# To properly tabularize the data, first zip both columns together
# This takes the 0th, 1st, ith, etc., element of each list, combines them into a pair, and returns a list of pairs
# Example: zip([1, 2, 3], [4, 5, 6]) = [(1, 4), (2, 5), (3, 6)]
zip = arrays_zip("time", "temperature_2m", "precipitation")

# Explode takes a list and returns a "column" with one row for each element
# "Column", or rather, an expression/execution step that can be used to select a column
# Alias is just the column name, "col" by default
df_exploded = dataframe.select(
    col("city"),
    explode(zip).alias("weather_record")
)

display(df_exploded)

# You must wrap the exploded table in a select statement to get the columns to show up (creates a new data frame with the column)
# Only then can you select the columns' values to create a new data frame with the columns, which will be named as the key (time, temperature_2m)
df_bronze_all_cities = df_exploded.select(
    col("city"),
    col("weather_record.time"),
    col("weather_record.temperature_2m"),
    col("weather_record.precipitation")
)

# Adds a column with the current timestamp
df_bronze_all_cities = df_bronze_all_cities.withColumn("ingested_at", current_timestamp())

display(df_bronze_all_cities)

In [0]:
from delta.tables import DeltaTable

target_table_name = "weather_project.north_texas_weather.bronze_hourly_multi_city"

if spark.catalog.tableExists(target_table_name):
    print("Found existing table. Merging new data...")

    target_delta = DeltaTable.forName(spark, target_table_name)

    # Incremental load (merge only new data)
    target_delta.alias("target") \
        .merge(
            source=df_bronze_all_cities.alias("source"),
            condition="target.city = source.city AND target.time = source.time" # Only merge based on these conditions
        ) \
        .whenMatchedUpdate(set={
            "temperature_2m": "source.temperature_2m",
            "precipitation": "source.precipitation",
            "ingested_at": "source.ingested_at"
        }) \
        .whenNotMatchedInsertAll() \
        .execute()

    print("Delta Merge successful! Duplicates blocked, new hours inserted.")

else:
    print("Table does not exist yet. Running first-time initial write layout...")

    df_bronze_all_cities.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("weather_project.north_texas_weather.bronze_hourly_multi_city")

    print("Initial baseline table established successfully!")